# Evolving Benchmarks — Evaluation as a Service (BYOA)

**Evaluate your own agent** on hosted, executable benchmarks over plain HTTP. You
**bring your own agent** (BYOA); the service hosts the *environment* and the
*grader*, so there is **no heavy environment to stand up** — no Docker gyms,
sandboxes, or datasets to download. The client is **`simple_agentic_evals`** — a
lightweight library for evaluating language models (inspired by
[openai/simple-evals](https://github.com/openai/simple-evals)), currently covering
**EOG** and **ALE**. The **service serves it itself**, so `pip install` needs no
PyPI account or repo checkout (its only dependency is `httpx`).

Every task is the same three-step loop:

```
create session   ->   your agent acts   ->   grade   ->   (session auto-freed)
```

## What the service gives you

- **Two executable benchmarks, one API:**
  - **EOG** (EnterpriseOps-Gym) — Dockerized enterprise apps. Your agent acts by
    calling **MCP tools** (proxied to a live gym and bound to your session's
    database) and is graded by hidden **SQL state verifiers**.
  - **ALE** (Agents' Last Exam) — a **file sandbox**. Your agent fetches input
    files, works in its own environment, and **submits an artifact**, graded by
    the task's own real `evaluate()`.
- **A built-in continual-learning axis** — every domain is *staged* into versions
  (`v1`, `v2`, …); the environment **evolves** as its tool set / skill library /
  agent pool **grows each stage**. Sweep the stages to measure how an agent copes.
- **Evolving resources you control** — for each task, request the evolving
  **tools / skills / agents** at `oracle` (gold), `accumulative` (the realistic
  growing set), or `none` (baseline).
- **Fully remote & self-cleaning** — everything rides one base URL; sessions are
  seeded on demand and torn down automatically.

## What's in this notebook

The agent here is a **real OpenAI model** — you'll be asked for your API key.

1. **Setup** — install the **`simple_agentic_evals`** SDK, connect, and enter your **OpenAI key**.
2. **EOG examples** — a provided harness acts via gym **MCP tools** across all three datasets:
   **tools** (`react_agent`, the reference ReAct agent), then **skills** and **agents**
   (`acp_codex_agent`, Codex over ACP). Every run reports **accuracy + latency + tokens**.
3. **ALE examples** — `acp_codex_agent` solves inside the Docker sandbox (solve + grade in one
   shot), across **tools / skills / agents**. (`react_agent` is EOG-only.)
4. **Continual learning** *(advanced)* — walk the evolving axis and score it with the SDK's
   bundled `ContinualMetrics` (ACC / BWT / FWT).

Each example also shows how to **list that dataset's evolving resource** — tool names,
`SKILL.md` bundles, or agent specs — with `client.resources(...)`.

## Taster — an agentic eval in a few lines

Install the SDK (one line — see **Setup** below), then let a provided harness run on a task
and grade it. `EvalClient()` already knows where to connect (the URL is baked into the wheel
the service ships), so there's no URL to wire up, and the environment is seeded and torn down
for you. Every dataset ships **`train`** and **`test`** splits — pass `split=` to pick one
(hold out `test` to report on). Every run reports **accuracy + latency + tokens**.

**Version-aware** — evaluate on a single evolving stage (`version=1`):

```python
from simple_agentic_evals import EvalClient, react_agent

client = EvalClient()                          # endpoint resolved for you (no URL to paste)

# Train / test splits for this stage:
train_ids = client.task_ids("evovling_tools", "eog", version=1, split="train", domain="hr")
test_ids  = client.task_ids("evovling_tools", "eog", version=1, split="test",  domain="hr")
print(f"v1: {len(train_ids)} train / {len(test_ids)} test tasks")

for task in client.tasks("evovling_tools", "eog", version=1, split="test", domain="hr", limit=5):
    with task:                                 # provisions a fresh environment
        run = react_agent(task, api_key="sk-...")            # the harness runs on the service
        print(task.task_id, task.grade().pass_rate, run.latency_s, run.total_tokens)
```

**Full dataset** — evaluate across *every* stage at once (`version="full"`), same split control.
`task.evaluate(...)` runs a harness then grades, returning all three metrics as an `EvalReport`:

```python
# The whole corpus (all stages), split into train / test:
test_ids = client.task_ids("evovling_tools", "eog", version="full", split="test", domain="hr")
print(f"full: {len(test_ids)} test tasks (across all stages)")

for task in client.tasks("evovling_tools", "eog", version="full", split="test", domain="hr", limit=5):
    with task:
        report = task.evaluate(agent="react", api_key="sk-...")   # run + grade in one call
        print(task.task_id, report.accuracy, report.latency_s, report.total_tokens)
```

That's the whole loop. The rest of this notebook just varies **what** you evaluate
(**tools / skills / agents**, on **EOG** or **ALE**) and **which** harness runs
(`react_agent` for EOG, `acp_codex_agent` for EOG + ALE).

# 1. Setup & connect

This notebook installs two small packages: **`openai`** (to drive a real agent) and
the **`simple_agentic_evals`** SDK — the client for the service (only dependency: `httpx`).
You need **no PyPI account and no repo checkout**: the service **ships its own client
wheel** at `GET /sdk`, so the connect step below `pip install`s it straight from the
service.

**You don't manage the service URL.** The setup cell reads it once from the
`EVAL_SERVICE_URL` env var (preconfigured by whoever hosts the service — on Colab, just
**Run all**), and from then on you simply call `EvalClient()` with no arguments. If the
endpoint enforces a key, set `EVAL_SERVICE_API_KEY` too.

You'll be **prompted for your OpenAI API key** — the key the reference agent uses to think.
Pre-set `OPENAI_API_KEY` in the environment or Colab secrets to skip the prompt.

Everything — **create / act / grade / teardown** — goes through the SDK over one endpoint
for **both** benchmarks: EOG tool calls reach the live gym (bound to your session's DB) and
ALE inputs/submissions travel the same way. You stand up no environment yourself.

In [ ]:
# openai powers the bring-your-own-agent examples in this notebook. (The simple_agentic_evals
# SDK — the client for the service — is installed in the "connect" step below: it is fetched
# FROM the service itself, which already knows its own endpoint, so you set no URL.)
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
print("openai ready  (simple_agentic_evals installs in the connect step below)")

In [ ]:
import os
from getpass import getpass

# ── Your OpenAI key — the key the agent harness uses to think ────────────────
# The provided harnesses (react_agent / acp_codex_agent) run on the SERVICE using
# this OpenAI key. Pre-set OPENAI_API_KEY (env var or Colab secrets) to skip the
# prompt. (You do NOT set a service URL: it is baked into the SDK the service ships,
# so `EvalClient()` connects with no arguments — see the next cell.)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
LLM_MODEL = os.environ.get("EVAL_LLM_MODEL", "gpt-4o-mini")

# ── Which slice of the benchmark to evaluate on ──────────────────────────────
# dataset:   evovling_tools | evovling_skills | evovling_agents
# benchmark: eog (Docker gyms, graded by SQL)  |  ale (file sandbox, graded on the artifact)
# domain:    eog only — hr | csm | itsm | calendar | email | teams | drive | hybrid | ...
# version:   the EVOLVING axis — v1 is the smallest tool universe; it grows each version.
DATASET   = "evovling_tools"
BENCHMARK = "eog"
DOMAIN    = "hr"
VERSION   = 1
SPLIT     = "test"

print("Model:", LLM_MODEL)
print(f"Cell: {DATASET}/{BENCHMARK}/{DOMAIN}/v{VERSION}/{SPLIT}")

In [ ]:
# ── Install the simple_agentic_evals SDK straight from the service ───────────
# The service ships its own client wheel at GET /sdk, so `pip install` works from
# anywhere you can reach the service — no PyPI account or repo checkout needed. The
# URL below is used ONLY to fetch that wheel; the wheel bakes the service endpoint
# in, so afterwards `EvalClient()` connects with NO url in your code.
import importlib, importlib.util, json, os, subprocess, sys, urllib.request

# Where to fetch the client wheel from (the hosted service). Override with
# $EVAL_SERVICE_URL if you run your own service.
_SERVICE_URL = os.environ.get("EVAL_SERVICE_URL", "https://idealist-unwritten-astronaut.ngrok-free.dev")


def _sdk_is_current() -> bool:
    # (Re)install unless a fresh-enough SDK is already importable. A stale, pre-installed
    # copy would otherwise shadow the service's wheel — it might miss `react_agent`
    # (older builds named it differently) or the background-poll support that long
    # ALE/Codex runs need — so we check the API *and* the version, not just presence.
    _MIN = (0, 10, 0)
    try:
        import simple_agentic_evals as _m
        if not (hasattr(_m, "react_agent") and hasattr(_m, "acp_codex_agent")):
            return False
        _v = tuple(int(x) for x in str(getattr(_m, "__version__", "0")).split(".")[:3])
        return _v >= _MIN
    except Exception:
        return False


if not _sdk_is_current():
    _req = urllib.request.Request(
        f"{_SERVICE_URL}/sdk", headers={"ngrok-skip-browser-warning": "true"}
    )
    _wheel = json.load(urllib.request.urlopen(_req))["path"]   # /sdk/simple_agentic_evals-<ver>.whl
    # --force-reinstall so a stale version is fully replaced; httpx is the only runtime dep.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
         "--no-deps", f"{_SERVICE_URL}{_wheel}"], check=True
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx"], check=True)
    # Drop any stale copy already imported into this kernel so the fresh install wins.
    for _k in [k for k in list(sys.modules)
               if k == "simple_agentic_evals" or k.startswith("simple_agentic_evals.")]:
        del sys.modules[_k]
    importlib.invalidate_caches()

from simple_agentic_evals import (   # the installed SDK — client + provided harnesses
    EvalClient, MCPSession, ServiceError,
    react_agent, acp_codex_agent, to_openai_tools,
)

# EvalClient() needs no URL: the service endpoint is baked into the wheel it ships,
# so the address never appears in your code (override via $EVAL_SERVICE_URL or
# EvalClient(base_url=...) if you must). If the endpoint enforces a key, set
# $EVAL_SERVICE_API_KEY and it is sent as `Authorization: Bearer <key>` on every call.
client = EvalClient()
print("client ready ->", client.health())

### Using the SDK

Everything below runs through the installed **`simple_agentic_evals`** SDK:
`EvalClient` (discovery, sessions, resources, grading), `MCPSession` (a minimal MCP
client to *act* on EOG tools), and `ServiceError`. It also **bundles the two provided
harnesses** — `react_agent` (EOG's reference ReAct agent) and `acp_codex_agent` (Codex
over ACP; EOG + ALE) — plus `to_openai_tools` (the MCP→OpenAI schema bridge for
bring-your-own-agent) and `ContinualMetrics`. Both harnesses run **on the service**
using the OpenAI `api_key` you pass, so nothing runs on your machine. The
bring-your-own-agent examples use a **real OpenAI model**, so we set up the `openai`
client next.

In [ ]:
from openai import OpenAI

# OpenAI client for the *bring-your-own-agent* examples (the EOG tool-loop and the
# ALE produce-an-artifact demos later). The provided harnesses (`react_agent` /
# `acp_codex_agent`) run on the SERVICE with the `api_key` you pass, so they need no
# local client — this `oai` is only for the BYOA sections.
oai = OpenAI()
print("OpenAI client ready; model =", LLM_MODEL)

### Discover what's available

You browse everything **through the SDK** — no raw HTTP:

- `client.benchmarks()` — the catalog: datasets → benchmarks → domains → versions (with
  train/test counts).
- `client.task_ids(dataset, benchmark, version, split, domain)` — the task ids for one slice.
- `client.tasks(...)` / `client.task(...)` — task handles to run.
- `client.resources(dataset, benchmark, …)` — a task's **evolving resource** (tool names,
  `SKILL.md` bundles, or agent specs), which each example below shows for its dataset.

The cell below defines a small `first_slice(dataset, benchmark)` helper (used later) that
returns the first `(domain, version, task_id)` that actually has a task, and then **lists the
evolving resource for one task in each dataset** — tool names, `SKILL.md` bundles, and agent
specs — with `client.resources(...)`. Same call for all three; `mode` picks `oracle` (gold) /
`accumulative` (growing) / `none`, and `include_content=True` returns the `SKILL.md` / agent-spec
bodies. (For a live EOG session, `task.mcp_session(server).list_tools()` enumerates the actual
callable tools too — see the *Bring your own agent* cell.)

In [ ]:
cat = client.benchmarks()
for ds in cat["datasets"]:
    for b in ds["benchmarks"]:
        doms = b["domains"]
        shown = ", ".join(
            f"{(d['domain'] or b['benchmark'])}(v1..v{max((v['version'] for v in d['versions']), default=0)})"
            for d in doms[:6]
        )
        print(f"{ds['dataset']:18s} {b['benchmark']:4s} [{b['kind']}]  {len(doms)} domain(s): {shown}")

print(f"\nTask ids in {DATASET}/{BENCHMARK}/{DOMAIN}/v{VERSION}/{SPLIT}:")
ids = client.task_ids(DATASET, BENCHMARK, VERSION, SPLIT, DOMAIN, limit=5)
for tid in ids:
    print("  ", tid)


def first_slice(dataset, benchmark):
    """First (domain, version, task_id) that actually has a task, for any dataset.

    Scans the catalog so the examples below work regardless of which domains /
    versions a dataset ships (eog carries domains; ale is flat -> domain is None).
    """
    for ds in client.benchmarks()["datasets"]:
        if ds["dataset"] != dataset:
            continue
        for b in ds["benchmarks"]:
            if b["benchmark"] != benchmark:
                continue
            for d in b["domains"]:
                dom = d["domain"] or None
                for v in sorted(x["version"] for x in d["versions"]):
                    got = client.task_ids(dataset, benchmark, v, "test", dom, limit=1)
                    if got:
                        return dom, v, got[0]
    return None, None, None


# List the evolving RESOURCE for a given task — the tool names (evovling_tools),
# SKILL.md bundles (evovling_skills), or agent specs (evovling_agents). It's the SAME
# call for all three: client.resources(...). `mode` picks which set you get: oracle
# (the task's gold set) | accumulative (the realistic growing set) | none (baseline).
print("\nEvolving resource for one task per dataset (mode=oracle):")
for ds in ("evovling_tools", "evovling_skills", "evovling_agents"):
    dom, ver, tid = first_slice(ds, "eog")
    if tid is None:
        print(f"  {ds:16s}: no eog task found"); continue
    r = client.resources(ds, "eog", ver, task_id=tid, split="test", domain=dom,
                         mode="oracle", include_content=False)
    print(f"  {ds:16s} [{dom} v{ver}]: kind={r['kind']:6s} count={r['count']:>2d}  "
          f"names={r['names'][:5]}")
# Tip: for a LIVE EOG session you can also enumerate the actual callable tools via
# task.mcp_session(server).list_tools() (see the BYOA cell); pass include_content=True
# to resources() to get the SKILL.md / agent-spec bodies, not just their names.

# 2. EOG examples — a real agent acts via MCP tools

**EOG** tasks run in a Dockerized enterprise gym. `POST /v1/sessions` provisions a
fresh per-session database and returns the **task prompts** + an **`action`** that
tells you how to act.

For EOG, `action.type == "mcp"` and `action.mcp_servers` is `[{name, url, headers}]`.
The agent **acts by calling MCP tools** on that `url` (sending the `headers` on every
call); the service proxies each call to the live gym and binds it to your session's
database. Then `grade()` runs the task's hidden SQL state verifiers.

We start with **`evovling_tools`** and the **reference ReAct agent** (`react_agent`) —
one task end to end, then the full loop. Then we run **`evovling_skills`** and **`evovling_agents`**
driven by **Codex over ACP** (`acp_codex_agent`). Each shows how to **list** its evolving
resource with `client.resources(...)`.

### The provided harness — `react_agent`

`react_agent` runs EnterpriseOps-Gym's **reference ReAct agent** on the task — **on the
service**: it reasons and calls the gym's MCP tools in a loop until it stops (or hits
`max_steps`), then returns an `AgentRun` carrying `latency_s` and token totals. Pass your
OpenAI `api_key`; it's the **same reference agent the benchmark uses**, so the score is
comparable. (`react_agent` is **EOG-only** — on an ALE task it raises a clear error; use
`acp_codex_agent` for ALE.)

Prefer your own agent? Hand `task.mcp_servers` to any MCP client and use the SDK's
**`to_openai_tools`** (the MCP→OpenAI schema bridge) — see the *Bring your own agent* cell
below. Here we bind our `api_key` + model to the provided harness once.

In [ ]:
from functools import partial

# The provided EOG ReAct harness. Bind our OpenAI api_key + model once; calling
# `agent(task)` then runs the reference ReAct loop on the SERVICE and returns an
# AgentRun. Swap in your own agent (see the BYOA cell) anywhere `agent(task)` appears.
agent = partial(react_agent, api_key=os.environ["OPENAI_API_KEY"], model=LLM_MODEL)

### One task, end to end

Provision one task, look at what the service returns, run the agent, and grade.

In [ ]:
# Grab the first task and start a session.
task = next(client.tasks(DATASET, BENCHMARK, VERSION, SPLIT, DOMAIN, limit=1))
task.start()

print("session_id :", task.session_id)
print("task_id    :", task.task_id)
print("action     :", task.action_type)
print("\nSYSTEM PROMPT (first 300 chars):\n", (task.system_prompt or "")[:300], "...")
print("\nUSER PROMPT:\n", task.user_prompt)
print("\nORACLE TOOLS (advisory hint):", task.oracle_tools)
print("\nMCP SERVERS the agent should act on:")
for s in task.mcp_servers:
    print("  ", s.name, "->", task.mcp_url(s))
    print("     headers:", s.headers)

In [ ]:
# Run the provided harness on the task we just provisioned. verbose="steps" prints a
# summary line + per-step thoughts / tool calls (name+args); use "full" to also print
# the raw final message, or include_trace=True to capture run.trace programmatically.
print("running the agent ...")
run = agent(task, verbose="steps")           # -> AgentRun(n_calls, steps, latency_s, total_tokens, ...)
print(f"\nagent made {run.n_calls} tool call(s); stopped: {run.stopped}.")
print(f"latency: {run.latency_s}s   tokens: {run.total_tokens} (in={run.input_tokens} out={run.output_tokens})")

In [ ]:
# Grade the current DB state against the task's hidden SQL verifiers, then free the DB.
result = task.grade()          # keep_alive=False -> the session is torn down for you
task.close()                   # idempotent; also frees the DB if grade didn't

print(f"pass_rate      : {result.pass_rate:.2f}  ({result.n_passed}/{result.n_total})")
print(f"overall_success: {result.overall_success}")
print("\nper-verifier:")
for v in result.per_verifier:
    print(f"  [{'PASS' if v['passed'] else 'FAIL'}] {v['name']}  "
          f"(expected {v.get('comparison_type','')} {v.get('expected')!r}, got {v.get('actual')!r})")

# All three metrics together: accuracy (from grading) + latency + tokens (from the run).
print(f"\nmetrics -> accuracy={result.pass_rate:.2f}  latency={run.latency_s}s  tokens={run.total_tokens}")

### The full eval loop

Putting it together: `with task:` provisions the session on enter and tears it down
on exit (so only the task you're running holds a live DB). The same provided harness
(`agent`, bound to `react_agent` above) runs on each task; everything else is just
bookkeeping. (Prefer your own agent? See the *Bring your own agent* section.)

In [ ]:
N = 3
scores = []
for task in client.tasks(DATASET, BENCHMARK, VERSION, SPLIT, DOMAIN, limit=N):
    with task:                       # seeds a fresh DB; torn down on block exit
        agent(task)                  # the real agent acts
        res = task.grade(keep_alive=True)
        scores.append(res.pass_rate)
        print(f"{task.task_id[:42]:42s}  pass_rate={res.pass_rate:.2f}")

if scores:
    print(f"\navg pass_rate over {len(scores)} tasks: {sum(scores)/len(scores):.3f}")

### One call for all three metrics — `task.evaluate(...)` + a trace

`task.evaluate(agent=...)` runs a harness then grades, returning an `EvalReport` with
**accuracy + latency + tokens** together (the raw `run` and `grade` are attached too). Ask for a
structured `trace` with `include_trace=True` (or `verbose="steps"`/`"full"`) to see per-step
thoughts, tool calls (name + args), and — for Codex on `evovling_agents` — sub-agent calls.

In [ ]:
# Run + grade in one call, then peek at the structured trace (per-step thought + tool calls).
task = next(client.tasks(DATASET, BENCHMARK, VERSION, SPLIT, DOMAIN, limit=1))
with task:
    report = task.evaluate(agent="react", api_key=os.environ["OPENAI_API_KEY"],
                           model=LLM_MODEL, include_trace=True, keep_alive=True)
    print(f"accuracy={report.accuracy:.2f}  latency={report.latency_s}s  tokens={report.total_tokens}")
    # report.run.trace: one dict per reason/act step (thought / tool_calls / tool_results / usage).
    for i, step in enumerate(report.run.trace[:3]):
        tools = ", ".join(c.get("name") or "" for c in step.get("tool_calls", []))
        print(f"  step {i}: tools=[{tools}]  thought={str(step.get('thought',''))[:80]!r}")

### Codex as the agent — `acp_codex_agent`

For **`evovling_skills`** and **`evovling_agents`**, the provided harness is **Codex over ACP**:

- **`acp_codex_agent(task, api_key=…, model=…)`** — runs Codex on the SERVICE with the OpenAI
  `api_key` you pass, in a **completion-sentinel loop with resume-on-stall** (the same
  `TASK_COMPLETE` protocol as the reference harness), and returns a `CodexRun` (`n_calls`,
  `episodes`, `stopped`, `latency_s`, `total_tokens`, `n_subagent_spawns`, `final_message`). For
  an `evovling_agents` task this is the reference **multi-agent orchestrator** (orchestrator +
  subagents), and `n_subagent_spawns` counts the sub-agent (agent) calls.

The loop is identical to `react_agent` — **provision → the harness runs → `grade()`** — because
grading reads the gym **database state**, not the transcript. The `api_key` here is your
**OpenAI** key (the agent uses it to think); it's separate from any `EvalClient(api_key=…)` that
authenticates to the service. `acp_codex_agent` also handles **ALE** (shown in section 3).

In [ ]:
import os
from simple_agentic_evals import acp_codex_agent

# Codex runs on the SERVICE with the OpenAI key you pass. Set OPENAI_API_KEY below.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    print("set OPENAI_API_KEY (your OpenAI key) to run Codex -> skipping.")
else:
    ids = client.task_ids(DATASET, BENCHMARK, VERSION, SPLIT, DOMAIN, limit=1)
    task = client.task(DATASET, BENCHMARK, VERSION, ids[0], split=SPLIT, domain=DOMAIN)
    with task:                                    # seed a fresh DB; freed on block exit
        # Codex acts on this session's gym in a completion loop; grading reads the
        # database state it leaves behind (not the transcript).
        run = acp_codex_agent(
            task,
            api_key=OPENAI_API_KEY,                   # OpenAI key the agent uses to think
            model=os.environ.get("CODEX_MODEL", "gpt-5-codex"),
            allowed_tools=task.oracle_tools or None,  # None -> full tool surface
            timeout_s=int(os.environ.get("CODEX_TIMEOUT_SEC", "600")),
            verbose="steps",
        )
        print(f"codex      : episodes={run.episodes} stopped={run.stopped} mcp_calls={run.n_calls}")
        print(f"metrics    : latency={run.latency_s}s tokens={run.total_tokens}")
        res = task.grade(keep_alive=True)             # grades the DB state Codex left behind
        print(f"pass_rate  : {res.pass_rate:.2f}  ({res.n_passed}/{res.n_total})  success={res.overall_success}")

### Skills & agents on EOG — driven by Codex

The same EOG loop works for the other two datasets — only the **evolving resource** changes:
`evovling_skills` ships `SKILL.md` bundles and `evovling_agents` ships agent specs. For each we
(1) **list** that resource with `client.resources(...)`, then (2) let **Codex** act on the task
(`acp_codex_agent`) — just pass your OpenAI `api_key`. Grading is identical: it reads the gym DB
state, not the transcript. (These run real Codex, so they're slower; they skip without a key.)

In [ ]:
# Skills & agents on EOG, driven by Codex. For each dataset: list the evolving resource
# (client.resources), then let Codex act on one task (acp_codex_agent), then grade.
if not os.environ.get("OPENAI_API_KEY"):
    print("set OPENAI_API_KEY to run Codex -> skipping this cell.")
else:
    for dataset in ("evovling_skills", "evovling_agents"):
        dom, ver, tid = first_slice(dataset, "eog")
        if tid is None:
            print(f"{dataset}: no eog task found -> skipping"); continue
        # (1) LIST this dataset's evolving resource for the task (oracle = gold set).
        r = client.resources(dataset, "eog", ver, task_id=tid, split="test",
                             domain=dom, mode="oracle")
        print(f"\n[{dataset} / eog v{ver} / {dom}] {tid}")
        print(f"  resource: kind={r['kind']} count={r['count']} names={r['names'][:5]}")
        # (2) Let Codex act on the task, with the stage resource attached.
        task = client.task(dataset, "eog", ver, tid, split="test", domain=dom,
                           resource_mode="oracle")
        with task:
            run = acp_codex_agent(
                task,
                api_key=os.environ["OPENAI_API_KEY"],           # OpenAI key the agent uses
                model=os.environ.get("CODEX_MODEL", "gpt-5-codex"),
                allowed_tools=task.oracle_tools or None,
                timeout_s=int(os.environ.get("CODEX_TIMEOUT_SEC", "600")),
            )
            res = task.grade(keep_alive=True)
            spawns = "" if run.n_subagent_spawns is None else f" agent_calls={run.n_subagent_spawns}"
            print(f"  codex: episodes={run.episodes} mcp_calls={run.n_calls}{spawns} "
                  f"latency={run.latency_s}s tokens={run.total_tokens} "
                  f"-> pass_rate={res.pass_rate:.2f} ({res.n_passed}/{res.n_total})")

# 3. ALE examples — Codex solves inside the sandbox

**ALE** tasks have no MCP: the task ships a prompt + **input files**, and the agent works
inside a Docker sandbox and produces a deliverable. ALE **requires a CLI agent harness**, so
the provided harness for ALE is **`acp_codex_agent`** — the service runs the Codex CLI agent
inside the sandbox, which **solves *and* grades** the task in one shot:

```
acp_codex_agent(task)  ->  (Codex works in the sandbox + the task's scorer runs)  ->  task.grade()  # inline score
```

- ALE is a **flat** layout — pass `domain=None` (just `dataset`, `version`, `split`).
- **Runs take several minutes.** The SDK submits the run as a **background job** and polls for the
  result (so it isn't cut off by a proxy/tunnel request timeout) — `acp_codex_agent(...)` is called
  the same way; add `verbose="steps"` for a liveness heartbeat.
- **Heavy tasks get a bigger budget automatically.** Left unset, `acp_codex_agent` picks `timeout_s`
  by benchmark: **900s** for EOG and **9000s** for ALE (reference parity — the agent phase gets
  7200s, matching `ale_run`). If a run still exceeds its budget you get a clean **502** (`ALE Codex
  run exceeded …s`), *not* a tunnel error — pass a larger `timeout_s` and re-run. The client waits
  `timeout_s + 300s`.
- `react_agent` does **not** work on ALE (it's EOG-only) — calling it raises a clear
  "ALE requires a CLI agent harness" error (shown two cells down).
- Prefer to drive it yourself? Bring your own agent: `task.inputs()` / `task.fetch_input(path)`
  fetch the staged inputs, you produce the artifact, then `task.submit_text(path, text)` and
  `task.grade()` (the *Bring your own agent* cell shows this).
- ALE ("Agents' Last Exam") tasks are **deliberately hard**, so a low score is normal. Some
  tasks need an environment we can't provide here (e.g. Windows-only) and return HTTP **501** —
  the cells below catch it.

In [ ]:
# ALE via the provided harness: the service runs the Codex CLI agent inside the sandbox,
# which solves AND grades the task. You pass your OpenAI api_key; there's nothing to submit.
try:
    # ALE is the flat layout -> domain=None. Target one task by id directly.
    task = client.task("evovling_tools", "ale", 1,
                       "legal/agora_governance_classify_instance_1", domain=None)
    with task:
        print("action:", task.action_type, "| deliverable:", task.output_path)
        print("inputs:", [f["path"] for f in task.inputs()][:4], "...")
        # Multi-minute run -> submitted as a background job + polled; verbose prints a heartbeat.
        # ALE auto-uses a reference-parity budget (timeout_s defaults to 9000s here; the agent
        # phase gets 7200s, like ale_run). Exceeding it returns a clean 502, not a tunnel error;
        # pass a bigger timeout_s to override. The client waits timeout_s + 300s.
        run = acp_codex_agent(task, api_key=os.environ["OPENAI_API_KEY"],
                              model=os.environ.get("CODEX_MODEL", "gpt-5-codex"),
                              verbose="steps")
        res = task.grade(keep_alive=True)          # returns the inline sandbox score
        print(f"latency: {run.latency_s}s")
        print(f"pass_rate: {res.pass_rate}  success: {res.overall_success}")
        print("(ALE tasks are deliberately hard; a low score here is normal.)")
except ServiceError as e:
    # needs_sandbox (501) -> the ALE Docker sandbox isn't available on this server.
    note = "needs the ALE Docker sandbox, not available here" if e.needs_sandbox else e.detail
    print("ALE run/grade unavailable here:", e.status_code, "-", note)

### `react_agent` is EOG-only — ALE raises a clear error

ALE needs a **CLI agent harness**, so `react_agent` (the ReAct-over-MCP harness) can't run it.
Calling it on an ALE task raises immediately — *before* any work — telling you to use
`acp_codex_agent` instead.

In [ ]:
# react_agent on an ALE task raises the "CLI agent harness" error (no key/session needed).
ale_task = client.task("evovling_tools", "ale", 1,
                       "legal/agora_governance_classify_instance_1", domain=None)
try:
    react_agent(ale_task)                      # EOG-only harness -> refuses ALE up front
except ServiceError as e:
    print(f"react_agent on ALE -> {e.status_code}: {e.detail}")

### Skills & agents on ALE

ALE runs inside the **Docker sandbox** via `acp_codex_agent` (the Codex CLI agent solves + grades
in one shot). The three datasets still differ by their **evolving resource**: `evovling_skills`
ships `SKILL.md` bundles and `evovling_agents` ships agent specs (for `evovling_agents`,
`acp_codex_agent` runs the multi-agent orchestrator inside the sandbox). You **list** them the
same way (`client.resources(...)`) and attach them via `resource_mode`. Below we list the resource
for each and run `acp_codex_agent` on one task each. ALE tasks are deliberately hard and some need
a sandbox we can't host here (HTTP **501**), so we guard for it.

> **These runs take several minutes** (a full Docker sandbox + Codex solve + grade). The SDK
> submits the run as a **background job** on the service and polls for the result, so it isn't cut
> off by a proxy/tunnel request timeout — you still just call `acp_codex_agent(...)`. We pass
> `verbose="steps"` below to print a liveness heartbeat while it works.

In [ ]:
# Skills & agents on ALE: list the evolving resource, then run acp_codex_agent (the Codex
# CLI agent solves + grades inside the sandbox; ALE has no MCP, so react_agent won't work here).
if not os.environ.get("OPENAI_API_KEY"):
    print("set OPENAI_API_KEY to run Codex -> skipping this cell.")
else:
    for dataset in ("evovling_skills", "evovling_agents"):
        dom, ver, tid = first_slice(dataset, "ale")   # ALE is flat -> dom is None
        if tid is None:
            print(f"{dataset}: no ale task found -> skipping"); continue
        r = client.resources(dataset, "ale", ver, task_id=tid, mode="oracle")   # LIST the resource
        print(f"\n[{dataset} / ale v{ver}] {tid}")
        print(f"  resource: kind={r['kind']} count={r['count']} names={r['names'][:5]}")
        try:
            task = client.task(dataset, "ale", ver, tid, resource_mode="oracle")
            with task:
                # Long run -> background job + polled. ALE auto-uses a reference-parity budget
                # (timeout_s defaults to 9000s; agent phase 7200s), so most runs finish without a
                # manual override; exceeding it still returns a clean 502, not a 503.
                run = acp_codex_agent(task, api_key=os.environ["OPENAI_API_KEY"],
                                      model=os.environ.get("CODEX_MODEL", "gpt-5-codex"),
                                      verbose="steps")
                res = task.grade(keep_alive=True)
                print(f"  ran (latency={run.latency_s}s) -> pass_rate={res.pass_rate} success={res.overall_success}")
                print("  (ALE tasks are deliberately hard; a low score here is normal.)")
        except ServiceError as e:
            note = "needs the ALE Docker sandbox, not available here" if e.needs_sandbox else e.detail
            print(f"  run/grade unavailable: {e.status_code} - {note}")

# Bring your own agent (BYOA)

Don't want the provided harnesses? **Drive the task yourself** — the environment, sessions, and
grading are identical (grading reads the gym DB / runs the task's own scorer, never your
transcript). The service hosts the environment + grader; you own the agent loop.

- **EOG** — get the gym's MCP tools with `to_openai_tools(task.mcp_session(server).list_tools())`,
  run your own model ↔ tool loop against the gym, then `task.grade()`.
- **ALE** — `task.inputs()` / `task.fetch_input(path)` to read the staged inputs → your agent
  produces the deliverable → `task.submit_text(path, text)` → `task.grade()` runs the real
  `evaluate()`.

Both examples reuse the `oai` client and `LLM_MODEL` from Setup, so they need `OPENAI_API_KEY`.

In [ ]:
# BYOA on EOG: a hand-rolled OpenAI tool loop over the gym's MCP tools (no provided harness).
import json

if not os.environ.get("OPENAI_API_KEY"):
    print("set OPENAI_API_KEY to run the BYOA loop -> skipping.")
else:
    task = next(client.tasks(DATASET, BENCHMARK, VERSION, SPLIT, DOMAIN, limit=1))
    with task:                                            # provisions the gym + DB
        server = task.mcp_servers[0]
        mcp = task.mcp_session(server)                    # pre-authed MCP client (EOG acting)
        tools = to_openai_tools(mcp.list_tools())         # MCP schemas -> OpenAI tool specs
        messages = [
            {"role": "system", "content": task.system_prompt or ""},
            {"role": "user", "content": task.user_prompt or ""},
        ]
        for _ in range(8):                                # cap the reason/act loop
            resp = oai.chat.completions.create(model=LLM_MODEL, messages=messages, tools=tools)
            msg = resp.choices[0].message
            messages.append(msg.model_dump(exclude_none=True))
            if not msg.tool_calls:
                break
            for tc in msg.tool_calls:                     # execute each tool call on the gym
                args = json.loads(tc.function.arguments or "{}")
                out = mcp.call_tool(tc.function.name, args)
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(out)[:4000]})
        mcp.close()
        res = task.grade(keep_alive=True)                 # grading is identical to the harness path
        print(f"BYOA(EOG): pass_rate={res.pass_rate:.2f} ({res.n_passed}/{res.n_total})")

In [ ]:
# BYOA on ALE: fetch inputs -> your agent produces the deliverable -> submit -> grade (no MCP).
import json

try:
    task = client.task("evovling_tools", "ale", 1,
                       "legal/agora_governance_classify_instance_1", domain=None)
    with task:
        inputs = task.inputs()                            # staged input files for the task
        print("inputs:", [f["path"] for f in inputs][:4], "...")
        prompt_ctx = task.fetch_input(inputs[0]["path"]).decode("utf-8", "replace")[:6000] if inputs else ""
        # --- your agent turns the prompt + inputs into the deliverable; a real one would
        #     parse the inputs and reason. Here we just call the model once for a JSON answer. ---
        if os.environ.get("OPENAI_API_KEY"):
            msg = [{"role": "system", "content": task.system_prompt or ""},
                   {"role": "user", "content": (task.user_prompt or "") + "\n\n" + prompt_ctx}]
            artifact = oai.chat.completions.create(model=LLM_MODEL, messages=msg).choices[0].message.content or "{}"
        else:
            artifact = json.dumps({"answer": "REPLACE_WITH_YOUR_AGENT_OUTPUT"})
        out = task.output_path or "output/agent_output.json"
        task.submit_text(out, artifact)                   # write the deliverable the prompt asks for
        res = task.grade(keep_alive=True)                 # runs the task's real evaluate()
        print(f"BYOA(ALE): submitted -> pass_rate={res.pass_rate} success={res.overall_success}")
        print("(ALE tasks are deliberately hard; a low score here is normal.)")
except ServiceError as e:
    note = "needs a sandbox we can't provide here" if e.needs_sandbox else e.detail
    print("BYOA(ALE) grading unavailable:", e.status_code, "-", note)

# 4. Continual learning — the evolving axis *(advanced topic)*

> **Advanced.** If you just want to evaluate a model, sections 1–3 are enough. This
> section covers the *evolving* harness — the research angle most users can skip.

This is the heart of the benchmark. Every domain is **staged** into versions
(`v1`, `v2`, …) and its **resources evolve** — the set an agent can draw on
**grows each stage**. Evaluating an agent as the stages advance is a
**continual-learning** study.

Each dataset evolves a different resource. You choose how much of it your agent
gets with two knobs — **`version`** (a single stage `1, 2, …`, or **`"full"`** for
every task at once) and **`resource_mode`**:

| dataset | evolving resource | `oracle` | `accumulative` | `none` |
| --- | --- | --- | --- | --- |
| `evovling_tools` | MCP tool names | minimal gold set | **full set at the stage** | none |
| `evovling_skills` | `SKILL.md` bundles | gold skill bundle(s) | full set at the stage | none |
| `evovling_agents` | agent `.toml` + skill | this task's gold specialists | full pool at the stage | none |

- **`oracle`** = the skyline — only what this task needs.
- **`accumulative`** = the realistic evolving setting — everything up to this stage (distractors included).
- **`none`** = baseline — the agent brings nothing.

The SDK **bundles the continual-learning metrics**: `ContinualMetrics` computes ACC,
backward/forward transfer, and forgetting from a results matrix — so you can run the
whole evolving study straight from this notebook, with no extra harness.

Below we (1) **inspect** the evolving resource per mode, (2) watch it **grow**
across stages, and (3) run a small **per-stage sweep** with the bundled `agent`,
then score it with `ContinualMetrics` — the learning-curve diagonal.

### Inspect the evolving resource

Two ways to see it:
- **`client.resources(...)`** — the names, and (for skills/agents) the actual
  `SKILL.md` / `.toml` **contents**, without starting a session.
- **`resource_mode=` on `client.tasks(...)`** — the same set rides on each session
  as `task.resources`, ready to wire into your agent as its allowlist.

In [ ]:
# --- 1) Inspect the evolving resource for a task, in each mode -----------------
# evovling_tools -> tool names; evovling_skills -> SKILL.md; evovling_agents -> .toml
for mode in ("oracle", "accumulative", "none"):
    r = client.resources("evovling_agents", "ale", version=2, mode=mode,
                         task_id="business_finance/digital_marketing_audience_segmentation_1")
    print(f"[agents v2 / {mode:12s}] kind={r['kind']} count={r['count']} names={r['names'][:5]}")

# Skills: the SKILL.md content comes back inline, ready to mount into your agent.
sk = client.resources("evovling_skills", "ale", version=3, mode="oracle",
                       task_id="legal/legal_dr_fees_01")
if sk.get("items"):
    md = next((f for f in sk["items"][0]["files"] if f["path"].endswith("SKILL.md")), None)
    print(f"\n[skills oracle] {sk['count']} skill(s); first bundle: {sk['items'][0]['name']}")
    if md and md.get("content"):
        print("  SKILL.md head:\n   ", md["content"].splitlines()[0][:80])

# --- 2) Stage = "full": every task across all stages, + the whole universe -----
all_ids = client.task_ids("evovling_tools", "ale", version="full", split="test")
print(f"\n[full stage] {len(all_ids)} tasks across ALL versions (vs a single stage)")
univ = client.resources("evovling_tools", "ale", version="full", mode="accumulative",
                         task_id=all_ids[0], include_content=False)
print(f"[full + accumulative] whole tool universe = {univ['count']} tools")

# --- 3) Pick a mode for a whole eval loop: it rides on task.resources ----------
for task in client.tasks("evovling_tools", "eog", domain="hr", version=2,
                         split="test", resource_mode="accumulative", limit=1):
    with task:
        res = task.resources or {}
        print(f"\n[session] {task.task_id}\n  resource_mode={res.get('mode')} "
              f"{res.get('kind')} count={res.get('count')} -> use as your agent's allowlist")
        # my_agent.run(task.system_prompt, task.user_prompt, task.mcp_servers, allow=res['names'])

### Watch the resource grow across stages

Walk a domain's versions and print how the **accumulative** set expands — this
growth *is* the evolving axis your agent has to keep up with.

In [ ]:
# The evolving axis, made concrete: pick a dataset + domain and walk its stages.
# The accumulative resource (tools here) grows every version — v1 is smallest.
DS_CL, BM_CL, DOM_CL = "evovling_tools", "eog", "hr"

cat = client.benchmarks()
versions = []
for ds in cat["datasets"]:
    if ds["dataset"] != DS_CL:
        continue
    for b in ds["benchmarks"]:
        if b["benchmark"] != BM_CL:
            continue
        for d in b["domains"]:
            if (d["domain"] or None) == DOM_CL:
                versions = sorted(v["version"] for v in d["versions"])

print(f"{DS_CL} / {DOM_CL}: stages -> {versions}\n")
print(f"{'stage':6s} {'#tasks':>7s}   accumulative resource (the evolving set)")
for v in versions:
    ids = client.task_ids(DS_CL, BM_CL, v, SPLIT, DOM_CL)
    if not ids:
        continue
    r = client.resources(DS_CL, BM_CL, v, task_id=ids[0], split=SPLIT, domain=DOM_CL,
                         mode="accumulative", include_content=False)
    print(f"  v{v:<4d} {len(ids):>7d}   {r['count']:>3d} {r['kind']}")

### A basic continual-learning sweep

Evaluate at each stage with that stage's `accumulative` resource, using the bundled
`agent`, then score the per-stage results with `ContinualMetrics`. These scores are
the learning-curve **diagonal** `R[k][k]`; filling the full `R[k][j]` matrix
(re-evaluating a memory-carrying agent on earlier stages) is what yields BWT / FWT.
(Point `DS_CL` at `evovling_skills` / `evovling_agents` for the other two resources.)

In [ ]:
# Evaluate the bundled agent at EACH stage with that stage's accumulative resource,
# then score the diagonal with the SDK's ContinualMetrics (ACC / BWT / FWT / forgetting).
from simple_agentic_evals import ContinualMetrics, StageResult

stages = versions[:3]                        # first few stages, to keep it quick
metrics = ContinualMetrics(num_stages=len(stages))
for k, v in enumerate(stages):
    stage_scores = []
    for task in client.tasks(DS_CL, BM_CL, version=v, split=SPLIT, domain=DOM_CL,
                             resource_mode="accumulative", limit=2):
        with task:
            agent(task)                      # the real agent acts, using this stage's tools
            stage_scores.append(task.grade(keep_alive=True).pass_rate)
    acc = sum(stage_scores) / len(stage_scores) if stage_scores else 0.0
    # Record the diagonal cell R[k][k]: this stage's agent on this stage's tasks.
    metrics.record(StageResult(eval_stage=k, adapt_stage=k,
                               num_tasks=len(stage_scores), success_rate=acc))
    print(f"stage v{v}: avg pass_rate = {acc:.2f}  over {len(stage_scores)} task(s)")

print()
print(metrics.print_report())                # per-stage ACC + final ACC (diagonal-only here)

# Recap & what to try next

You've seen the whole surface: **Setup** → **EOG** (a real agent acts via MCP,
graded by SQL) → **ALE** (submit an artifact, graded by `evaluate()`) →
**Continual learning** (sweep the evolving axis). The loop is always
`create → act → grade → auto-freed`.

- **Change the slice.** Edit `DATASET` (`evovling_tools` / `evovling_skills` /
  `evovling_agents`), `BENCHMARK` (`eog` / `ale`), `DOMAIN`, `VERSION`, or `SPLIT`.
  Call `client.benchmarks()` to see what's available; pass `version="full"` to run
  every stage at once.

- **Choose the resource.** Set `resource_mode` (`oracle` | `accumulative` | `none`)
  on `client.tasks(...)`, and inspect it any time with `client.resources(...)`.

- **Every run reports three metrics.** `react_agent` / `acp_codex_agent` return `latency_s` and
  `total_tokens`; `grade()` returns accuracy (`pass_rate`). `task.evaluate(agent=...)` bundles all
  three into an `EvalReport` — pick whichever you care about. Pass `verbose="steps"` (or `"full"`)
  / `include_trace=True` to see per-step thoughts, tool calls, and sub-agent calls.

- **Two provided harnesses, run on the service.** `react_agent` (the reference ReAct agent over
  gym MCP tools; **EOG only**) and `acp_codex_agent` (Codex over ACP; the multi-agent orchestrator
  for `evovling_agents`, and the ALE harness) — all you pass is your OpenAI `api_key`; nothing runs
  on your machine. `react_agent` on an ALE task raises a clear "ALE requires a CLI agent harness —
  use acp_codex_agent" error.

- **Long runs poll automatically.** An ALE Docker+Codex solve (or a slow EOG Codex loop) can take
  several minutes; the harness helpers submit the run as a **background job** on the service and
  poll for the result, so it's never cut off by a proxy/tunnel request timeout — your code is
  unchanged (add `verbose="steps"` for a heartbeat, raise `timeout_s` for very long tasks).

- **Prefer your own agent?** Hand `task.mcp_servers` to any MCP client (`to_openai_tools` bridges
  the tool schemas) for EOG, or drive ALE with `task.inputs()` → produce → `task.submit()` →
  `task.grade()`. Sessions, environment, and grading stay the same; grading reads the gym state,
  not the transcript. (See the *Bring your own agent* section.)

- **It's just the SDK.** Everything here — the client (`EvalClient`, `MCPSession`,
  `ServiceError`) **and** the harnesses + metrics (`react_agent`, `acp_codex_agent`,
  `to_openai_tools`, `ContinualMetrics`) — is the installed **`simple_agentic_evals`** package.
  The service **serves its own wheel** at `GET /sdk`, and the endpoint is baked in, so
  `EvalClient()` needs no URL — drop this loop straight into your own code.

- **Run the full CL study.** Sweep every stage/mode with the bundled `agent` and feed
  the results matrix to `ContinualMetrics` for ACC / BWT / FWT / forgetting — the same
  metrics the research runs report, computed client-side against this service.

- **Cleanup is automatic.** A session is freed on `grade()` (unless
  `keep_alive=True`) and on `DELETE`; anything you forget is cleaned up for you.

The **complete API reference** for `simple_agentic_evals` is below.

## API reference — `simple_agentic_evals`

Everything the SDK exposes, systematically. Import surface:

```python
from simple_agentic_evals import (
    EvalClient, Task, GradeResult, EvalReport, McpServer, MCPSession, ServiceError,
    react_agent, acp_codex_agent,                    # provided harnesses (run on the service)
    AgentRun, CodexRun,
    to_openai_tools, sanitize_tool_schema,           # MCP -> OpenAI bridge (bring your own agent)
    ContinualMetrics, StageResult,                   # continual-learning metrics
)
```

Every run returns three metrics you can pick from: **accuracy** (`grade().pass_rate`),
**latency** (`run.latency_s`), and **tokens** (`run.total_tokens`). `Task.evaluate(agent=...)`
bundles all three into an `EvalReport`.

### Selector vocabulary (shared across methods)

These identify *which slice* of the benchmark you evaluate on:

| arg | type | values / meaning |
| --- | --- | --- |
| `dataset` | str | `evovling_tools` · `evovling_skills` · `evovling_agents` — which evolving corpus. |
| `benchmark` | str | `eog` (EnterpriseOps-Gym: Docker gyms, graded by SQL) · `ale` (Agents' Last Exam: file sandbox, graded by the task's own `evaluate()`). |
| `version` | int \| str | Evolving stage: `1, 2, …` for one stage, or `"full"` for every task across all stages. |
| `split` | str | `test` (default) · `train`. |
| `domain` | str \| None | EOG only: `hr`, `csm`, `itsm`, `calendar`, `email`, `teams`, `drive`, `hybrid`, … ALE is flat → `None`. |
| `resource_mode` | str \| None | Evolving resource to attach: `oracle` (gold minimal) · `accumulative` (realistic growing set) · `none` (baseline). `None` = don't attach. |
| `task_id` | str | A specific task id (from `task_ids()` / `tasks()`). |
| `limit`, `offset` | int | Page the task list. |

### `EvalClient` — entry point

```python
client = EvalClient()   # no URL needed — the endpoint is resolved for you (see base_url)
# full signature:
client = EvalClient(base_url=None, timeout=1800.0, api_key=None, extra_headers=None)
```

| param | default | meaning |
| --- | --- | --- |
| `base_url` | `None` | Service endpoint. Left unset it resolves automatically: `$EVAL_SERVICE_URL` → the URL baked into the wheel the service serves → `http://localhost:8077`. The address is resolved for you, so you normally just call `EvalClient()`. |
| `timeout` | `1800.0` | Per-request timeout (s); seeding a gym can be slow. |
| `api_key` | `None` | Service key; if unset, falls back to `$EVAL_SERVICE_API_KEY`. When present, sent as `Authorization: Bearer <key>` on every call (incl. MCP acting). |
| `extra_headers` | `None` | Extra headers merged into every request (`ngrok-skip-browser-warning` is always sent). |

| method | returns | what it does |
| --- | --- | --- |
| `.health()` | dict | Service status: `ok`, active sessions, TTL, ALE-docker availability. |
| `.benchmarks()` | dict | Catalog of datasets / benchmarks / domains / versions. |
| `.task_ids(dataset, benchmark, version, split="test", domain=None, limit=None, offset=0)` | list[str] | Task ids for a slice. |
| `.tasks(dataset, benchmark, version, split="test", domain=None, resource_mode=None, limit=None, offset=0)` | Iterator[`Task`] | Lazy task handles — no session until `start()` / `with`. |
| `.task(dataset, benchmark, version, task_id, split="test", domain=None, resource_mode=None)` | `Task` | One task by id. |
| `.resources(dataset, benchmark, version, task_id=None, split="test", domain=None, mode=None, include_content=True)` | dict | Inspect the evolving resource **without a session**; `mode` = `oracle`\|`accumulative`\|`none`; `include_content=True` returns `SKILL.md`/`.toml` bodies. |
| `.close()` | — | Close the HTTP client. Also a context manager: `with EvalClient(...) as client:`. |

### `Task` — one benchmark task

Context manager: `with task:` (or `task.start()`) seeds the environment; exit / `task.close()` frees the gym DB. `grade()` scores the current state.

**Attributes** (populated after `start()`):

| attribute | type | meaning |
| --- | --- | --- |
| `task_id` | str | Task identifier. |
| `session_id` | str \| None | Live session id (`None` before start). |
| `system_prompt`, `user_prompt` | str | Prompts to hand your agent. |
| `oracle_tools` | list[str] | Gold tool names for the task (EOG). |
| `resources` | dict | Evolving resource attached via `resource_mode`: `{kind, mode, count, names, …}`. |
| `action_type` | str | `"mcp"` (EOG) or `"sandbox"` (ALE). |
| `mcp_servers` | list[`McpServer`] | Gym MCP endpoints to act on (EOG). |
| `input_files` | list[dict] | Declared input files (ALE). |
| `output_path` | str | Deliverable path the prompt asks for (ALE). |
| `sandbox` | dict | Raw ALE action payload (`gradable`, `grading`, `output_dir`, …). |

**Methods:**

| method | returns | what it does |
| --- | --- | --- |
| `.start()` / `.close()` | `Task` / — | Provision / tear down the session (prefer `with task:`). |
| `.mcp_url(server)` | str | Absolute MCP endpoint for a gym server (proxied via the service). |
| `.mcp_session(server, timeout=60.0)` | `MCPSession` | Pre-authed MCP client for `server` (EOG acting). |
| `.inputs()` | list[dict] | List staged input files (ALE). |
| `.fetch_input(rel_path)` | bytes | Download one staged input file (ALE). |
| `.submit(files)` | list[str] | Submit artifact(s): `{"path","content"}` (text) or `{"path","content_b64"}` (binary). |
| `.submit_text(path, text)` | list[str] | Submit a single text artifact. |
| `.grade(keep_alive=False)` | `GradeResult` | Grade current state; frees the session unless `keep_alive=True`. |
| `.evaluate(agent="acp_codex", *, keep_alive=False, **kw)` | `EvalReport` | Run a harness (`"react"` / `"acp_codex"`) then grade, bundling accuracy + latency + tokens. Extra kwargs pass through to the harness (`api_key`, `model`, `verbose`, `include_trace`, …). ALE + `"react"` raises the CLI-harness error. |

### `GradeResult`

| field | type | meaning |
| --- | --- | --- |
| `task_id` | str | The graded task. |
| `pass_rate` | float | Fraction of verifiers passed, `0.0`–`1.0`. |
| `overall_success` | bool | Whether the task counts as solved. |
| `n_passed`, `n_total` | int | Verifiers passed / total. |
| `per_verifier` | list[dict] | Per-verifier detail. |
| `raw` | dict | Full server response. |

### `EvalReport` — accuracy + latency + tokens together

Returned by `Task.evaluate(agent=...)`; bundles the three metrics so you can pick what to report.

| field | type | meaning |
| --- | --- | --- |
| `task_id` | str | The evaluated task. |
| `agent` | str | Which harness ran: `"react"` \| `"acp_codex"`. |
| `accuracy` | float | `grade.pass_rate`, `0.0`–`1.0`. |
| `overall_success` | bool | Whether the task counts as solved. |
| `latency_s` | float \| None | Wall-clock seconds for the harness run. |
| `total_tokens` | int \| None | Total tokens used by the run (`None` if unavailable, e.g. some ALE runs). |
| `input_tokens`, `output_tokens` | int \| None | Prompt / completion token split when available. |
| `run` | `AgentRun` \| `CodexRun` | The raw run result (trace, per-step detail, …). |
| `grade` | `GradeResult` | The raw grade result (per-verifier detail, …). |

### `McpServer`

| field | type | meaning |
| --- | --- | --- |
| `name` | str | Gym / server name. |
| `url` | str | Absolute MCP URL. |
| `path` | str | Relative proxy path on the service (preferred; joined with `base_url`). |
| `headers` | dict | Per-server headers (optional; the proxy injects the DB binding). |
| `transport` | str | `streamable_http`. |

### `MCPSession` — minimal MCP client (EOG acting)

```python
mcp = task.mcp_session(server)      # or: MCPSession(url, headers=None, timeout=60.0, api_key="")
```

| method | returns | what it does |
| --- | --- | --- |
| `.list_tools()` | list[dict] | Tools the gym exposes (`initialize` + `tools/list`). |
| `.call_tool(name, arguments=None)` | dict | Invoke a tool (`tools/call`). |
| `.close()` | — | Close the client. Also a context manager. |

### `ServiceError` — raised on any non-2xx response

| attribute | type | meaning |
| --- | --- | --- |
| `status_code` | int | HTTP status code. |
| `detail` | str | Server-provided error detail. |
| `url` | str | The request URL. |
| `needs_sandbox` | bool | `True` when `status_code == 501` — an ALE task whose grader needs a full sandbox not available here. |

Handle service errors without importing `httpx`:

```python
try:
    res = task.grade()
except ServiceError as e:
    print(e.status_code, "-", "needs sandbox" if e.needs_sandbox else e.detail)
```

### Provided harnesses — `react_agent`, `acp_codex_agent`

Two ready-made harnesses so you don't reimplement the loop — both run **on the service**; you just
pass your OpenAI `api_key`. Every run returns `latency_s` and `total_tokens` alongside the run
detail, and `grade()` gives accuracy — or use `Task.evaluate(...)` to get all three at once. Pass
`verbose="steps"`/`"full"` or `include_trace=True` for per-step thoughts, tool calls, and
sub-agent calls.

```python
run = react_agent(task, api_key="sk-...", model=None, max_steps=None,
                  restrict_to_selected_tools=False, timeout_s=1800.0,
                  verbose=False, include_trace=False)           # EOG only
run = acp_codex_agent(task, api_key="sk-...", model=None, allowed_tools=None,
                      max_episodes=4, timeout_s=None,   # None -> 900s (EOG) / 9000s (ALE)
                      verbose=False, include_trace=False)        # EOG + ALE
```

| function | key params | returns | what it does |
| --- | --- | --- | --- |
| `react_agent(task, *, api_key=None, …)` | `model=None`, `max_steps=None`, `restrict_to_selected_tools=False`, `timeout_s=1800`, `verbose=False`, `include_trace=False` | `AgentRun` | Runs EnterpriseOps-Gym's **reference ReAct agent** on the service (reason ↔ gym MCP `tools/call` until it stops). Provisions the session if needed; does **not** close it. **EOG only** — on an ALE task it raises "ALE requires a CLI agent harness — use acp_codex_agent". |
| `acp_codex_agent(task, *, api_key=None, …)` | `model=None`, `allowed_tools=None`, `mcp_only=None`, `transport=None`, `max_episodes=4`, `timeout_s=None` (900s EOG / 9000s ALE), `require_completion=True`, `completion_sentinel="TASK_COMPLETE"`, `verbose=False`, `include_trace=False` | `CodexRun` | Runs **Codex over ACP** on the service in a completion-sentinel loop with resume-on-stall (the reference multi-agent orchestrator for `evovling_agents`). On **ALE** it runs the Codex CLI agent inside the Docker sandbox (solve + grade in one shot). Provisions the session; does **not** close it. |

`AgentRun`: `n_calls` (gym tool calls), `steps` (reason/act iterations), `stopped` (`"done"` | `"error"`), `completed`, `latency_s`, `total_tokens`, `input_tokens`, `output_tokens`, `trace`, `final_message`, `tools_used`, `messages`.

Deprecated aliases `run_eog_agent` → `react_agent` and `run_codex_agent` → `acp_codex_agent` still
work for one release (they emit a `DeprecationWarning`); `run_ale_agent` was removed (ALE now runs
server-side via `acp_codex_agent`).

### MCP → OpenAI bridge — `to_openai_tools`, `sanitize_tool_schema`

The schema rewriter that lets any OpenAI-style model call the gym's MCP tools
(OpenAI rejects a top-level `anyOf`/`oneOf`/`allOf`/`enum`/`not`). Use it directly if
you build your own agent instead of `react_agent`.

```python
mcp   = task.mcp_session(task.mcp_servers[0])
tools = to_openai_tools(mcp.list_tools())      # -> [{\"type\":\"function\",\"function\":{…}}]
```

| function | returns | what it does |
| --- | --- | --- |
| `to_openai_tools(mcp_tools, *, max_desc=1024)` | list[dict] | Convert MCP `tools/list` descriptors into OpenAI function-tool specs; strips forbidden top-level keywords and appends the constraint as a description hint. |
| `sanitize_tool_schema(schema)` | `(schema, hints)` | The underlying single-schema rewrite: cleaned object schema + human-readable constraint hints. |

### Continual-learning metrics — `ContinualMetrics`, `StageResult`

Compute ACC / BWT / FWT / forgetting from an evolving sweep, client-side.

```python
m = ContinualMetrics(num_stages=n)
m.record(StageResult(eval_stage=k, adapt_stage=k, num_tasks=x, success_rate=acc))
report = m.compute()          # {\"ACC\":[…], \"final_ACC\":…, \"BWT\":…, \"FWT\":…, …}
print(m.print_report(report))
```

Convention: `R[k][j]` = score on stage-`j` tasks after adapting through stage `k`
(lower triangle, `j ≤ k`); rows = adapt stage (time), cols = eval stage. Record a
stage's no-memory baseline with `adapt_stage=-1` to enable FWT.

| member | meaning |
| --- | --- |
| `ContinualMetrics(num_stages)` | Holder for the results matrix. |
| `.record(StageResult)` | Add one `(eval_stage, adapt_stage)` cell. |
| `.compute()` → dict | `ACC` (per stage), `final_ACC`, `BWT`, `FWT`, `avg_forgetting`, `max_forgetting`, `results_matrix`. |
| `.print_report(report=None)` → str | Human-readable matrix + metrics. |
| `StageResult(eval_stage, adapt_stage, num_tasks=0, success_rate=0.0, …)` | One cell; `success_rate` is the mean score (e.g. `pass_rate`). |

### `acp_codex_agent` — details

Runs **Codex over ACP** on the service with your OpenAI `api_key`, in a completion-sentinel loop
with **resume-on-stall** (the same `TASK_COMPLETE` protocol as the reference harness). For an
`evovling_agents` task it runs the reference multi-agent orchestrator (orchestrator + subagents),
and `n_subagent_spawns` counts the sub-agent calls. On an **ALE** task it instead runs the Codex
CLI agent inside the Docker sandbox, which solves *and* grades in one shot (`grade()` returns that
inline score). Grading is otherwise unchanged — for EOG it reads the gym state, not the transcript.

```python
run = acp_codex_agent(task, api_key="sk-...", model="gpt-5-codex", verbose="steps")
print(run.n_calls, run.stopped, run.latency_s, run.total_tokens, task.grade().pass_rate)
```

Params: `api_key` (your OpenAI key), `model`, `allowed_tools` (non-empty ⇒ oracle: restrict to the
task's gold tools), `mcp_only` (disable Codex's non-MCP built-ins), `transport`, `max_episodes=4`,
`timeout_s=None` (defaults to 900s for EOG, 9000s for ALE), `require_completion=True`, `completion_sentinel="TASK_COMPLETE"`, `verbose=False`,
`include_trace=False`. Provisions the session; does **not** close/grade it (except ALE, which
grades inline).

`CodexRun`: `n_calls` (gym MCP calls), `n_exec` (shell), `episodes`, `stopped` (`"done"` | `"max_episodes"` | `"timeout"` | `"error"`), `completed`, `latency_s`, `total_tokens`, `input_tokens`, `output_tokens`, `n_subagent_spawns` (agent calls), `trace`, `thread_id`, `final_message`.

Happy evaluating!
